# 文章を生成しないAI「Jev」に、1文字ずつ選ばせて文章を生成させる

TypeSafe AI の判断専用モデル **Jev** は Noul / Choice / Score しか返せず、文字列は生成しません。
そこで Choice の選択肢を「ひらがな全部 + 句読点 + END」（英語版は a〜z + 記号 + SPACE + END）にして、
「ここまでの回答」を state に入れながら次の 1 文字を選ばせ、END が出るまでループします。

必要なもの: TypeSafe の API キー（https://console.typesafe.ai/settings/keys ・ early access）

使い方: 上から順に実行 → 最後のセルの `query` を書き換えて実行。

In [ ]:
# API キーの入力（Colab の場合は左の鍵アイコン「シークレット」に TYPESAFE_API_KEY を登録しておくと自動で読みます）
import os
from getpass import getpass

API_KEY = os.environ.get("TYPESAFE_API_KEY")
if not API_KEY:
    try:
        from google.colab import userdata  # type: ignore
        API_KEY = userdata.get("TYPESAFE_API_KEY")
    except Exception:
        pass
if not API_KEY:
    API_KEY = getpass("TYPESAFE_API_KEY: ")
print("API key:", "set" if API_KEY else "NOT set")

In [ ]:
import argparse
import json
import os
import random
import sys
import time
import urllib.error
import urllib.request

API_URL = "https://api.typesafe.ai/v1/systemone"
END = "END"
SPACE = "SPACE"  # 選択肢名に空白は使いにくいので別名にする


def build_vocab(lang: str) -> dict[str, str]:
    """Choice の criteria（選択肢名 -> 説明）を作る。"""
    if lang == "ja":
        chars = [chr(c) for c in range(0x3041, 0x3094)]  # ぁ〜ん
        chars += ["ー", "、", "。", "？"]
        vocab = {c: f"文字「{c}」" for c in chars}
        vocab[END] = "回答はここで完成。これ以上文字を追加しない"
    elif lang == "en":
        chars = [chr(c) for c in range(ord("a"), ord("z") + 1)]
        chars += [".", ",", "?", "'"]
        vocab = {c: f"the letter '{c}'" for c in chars}
        vocab[SPACE] = "a space character (word boundary)"
        vocab[END] = "The answer is complete. Do not add any more characters."
    else:
        raise ValueError(f"unknown lang: {lang}")
    return vocab


def build_request(lang: str, question: str, answer_so_far: str, vocab: dict[str, str], model: str) -> dict:
    if lang == "ja":
        state = {
            "question": question,
            "answer_so_far": answer_so_far,
            "rule": "回答はひらがなだけで 1 文字ずつ書く。answer_so_far は途中まで書いた回答。",
        }
        instructions = "answer_so_far に続けて書くべき次の 1 文字。回答が完成していれば END"
    else:
        state = {
            "question": question,
            "answer_so_far": answer_so_far,
            "rule": "Write the answer in lowercase letters, one character at a time. answer_so_far is the partial answer.",
        }
        instructions = "The next single character to append to answer_so_far. Choose END if the answer is complete."
    return {
        "state": state,
        "model": model,
        "questions": {"next": {"type": "choice", "instructions": instructions, "criteria": vocab}},
    }


def call_jev(payload: dict, api_key: str) -> dict:
    req = urllib.request.Request(
        API_URL,
        data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"HTTP {e.code}: {body}")


def to_char(option: str) -> str:
    return " " if option == SPACE else option


def generate(question: str, lang: str, api_key: str, model: str, max_steps: int, sample: bool, top_k: int) -> str:
    vocab = build_vocab(lang)
    answer = ""
    total_tokens = 0
    t_start = time.time()

    print(f"Q: {question}")
    print(f"vocab: {len(vocab)} options ({'sampling' if sample else 'greedy'})\n")

    for step in range(1, max_steps + 1):
        payload = build_request(lang, question, answer, vocab, model)
        t0 = time.time()
        res = call_jev(payload, api_key)
        ms = int((time.time() - t0) * 1000)
        total_tokens += res.get("usage", {}).get("input_tokens", 0)

        ans = res["answers"]["next"]
        probs: dict[str, float] = ans["probabilities"]
        ranked = sorted(probs.items(), key=lambda kv: kv[1], reverse=True)

        if sample:
            names = list(probs.keys())
            weights = [max(probs[n], 0.0) for n in names]
            pick = random.choices(names, weights=weights, k=1)[0] if sum(weights) > 0 else ans["choice"]
        else:
            pick = ans["choice"]

        top = "  ".join(f"{to_char(n)!r}:{p:.2f}" for n, p in ranked[:top_k])
        print(f"step {step:2d}  pick={to_char(pick)!r:6}  conf={ans.get('confidence', 0):.2f}  {ms:4d}ms  top{top_k}: {top}")

        if pick == END:
            break
        answer += to_char(pick)
        print(f"         so far: 「{answer}」")

    print(f"\nA: 「{answer}」")
    print(f"({step} steps, {total_tokens} input tokens, {time.time() - t_start:.1f}s, model={res.get('model')})")
    return answer


## 日本語版（ひらがな 84 文字 + 、。？ + END の 88 択）

In [ ]:
query = "日本の首都はどこですか？"
generate(query, lang="ja", api_key=API_KEY, model="jev-latest", max_steps=20, sample=False, top_k=3)

## 英語版（a〜z + . , ? ' + SPACE + END の 32 択）

In [ ]:
query = "What is the capital of Japan?"
generate(query, lang="en", api_key=API_KEY, model="jev-latest", max_steps=20, sample=False, top_k=3)

## おまけ: greedy ではなく確率でサンプリングする

In [ ]:
import random
random.seed(1)
generate("日本の首都はどこですか？", lang="ja", api_key=API_KEY, model="jev-latest", max_steps=20, sample=True, top_k=3)